# Setup

In [1]:
import json

from torch.cuda import is_available as is_cuda_available
import numpy as np
from lighthouse.models import CGDETRPredictor
from utils.helper_methods import split_video_ffmpeg, parse_timestamps, extract_clips, adjust_video_speed
import pandas as pd
import os

C:\Users\denve\miniconda3\envs\moment-retrieval-and-video-summarization\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model weights can be downloaded from https://zenodo.org/records/13960580
Checkpoint file should correspond to the feature selected in predictor

For slowfast video encoding features additional weights should be downloaded. They can be found in lighthouse GitHub page (https://github.com/line/lighthouse/tree/main) or downloaded from [here](https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl)

In [2]:
device = "cuda" if is_cuda_available() else "cpu"
device

'cuda'

In [3]:
# model = CGDETRPredictor('model_weights/clip_cg_detr_qvhighlight.ckpt', device=device,
#                         feature_name='clip')

model = CGDETRPredictor('model_weights/clip_slowfast_cg_detr_qvhighlight.ckpt', device=device,
                        feature_name='clip_slowfast', slowfast_path="model_weights/SLOWFAST_8x8_R50.pkl", pann_path=None)

# Video preparation
CG-DETR model has a major limitation -- it doesn't allow videos longer than 150 seconds. We will try several approaches to this problem

In [4]:
# # Load events which should be retrieved
# events = pd.read_csv("Annotations/7-TVSum-video_25-i6324060.txt")[['annotation', 'timestamp']]
# events['timestamp'].replace(np.nan, None)
# events['timestamp'] = events['timestamp'].map(parse_timestamps)

In [5]:
with open("Descriptions/video_25.json", 'r') as f:
    events_json = json.load(f)

events_full = pd.json_normalize(events_json, record_path=['llm_results'])[['prompt_type', 'vlm_source', 'vlm_frame_count', 'parsed_events']].head()
events = events_full['parsed_events'].to_frame()
events['timestamps'] = np.nan

In [6]:
# set the original video path
og_video_path = "TVSum/video_25.mp4"

## Splitting
The most obvious way is splitting videos in chunks. We cut the original video in segments no longer than 150 seconds and then process them seperately

In [7]:
output_path = "processed_videos/video_25_chunks"
split_video_ffmpeg(og_video_path, output_path, 150)

Splitting TVSum/video_25.mp4 into 150s chunks...
Chunks are successfully saved into 'processed_videos/video_25_chunks' folder.


'processed_videos/video_25_chunks'

In [8]:
results = []
for row in events.iterrows():
    results_per_entry = [] # results for every event list

    queries, true_timestamps = row[1]
    for query in queries:
        results_per_chunk = [] # results for every query

        print(f"Query: {query}")
        print(f"True timestamp: {true_timestamps}")

        for index, chunk in enumerate(os.listdir(output_path)):
            if chunk.endswith('.mp4'):
                video_path = output_path + '/' + chunk
                video = model.encode_video(video_path)
                prediction = model.predict(query, video)
                results_per_chunk.append((index, prediction))

                print("\tChunk:", index)
                print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0]}, {prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")

        print('-' * 50)

        results_per_entry.append(results_per_chunk)

    results.append(results_per_entry)

Query: A man wearing headphones takes pictures with his camera while standing next to another person who has earphones plugged into it.
True timestamp: nan
	Chunk: 0
		Closest Moment: 16.2733, 26.9137, Corresponding Confidence score: 0.9965
	Chunk: 1
		Closest Moment: 86.394, 100.0044, Corresponding Confidence score: 0.9997
--------------------------------------------------
Query: Two young men walk down an alleyway lined with shops, one wearing sunglasses and carrying a backpack, the other in orange shorts.
True timestamp: nan
	Chunk: 0
		Closest Moment: 38.0969, 64.0586, Corresponding Confidence score: 0.983
	Chunk: 1
		Closest Moment: 56.5158, 68.892, Corresponding Confidence score: 0.9992
--------------------------------------------------
Query: The two young men pass by parked cars before entering a building where they appear to be conversing or taking photos together.
True timestamp: nan
	Chunk: 0
		Closest Moment: 74.9577, 84.7932, Corresponding Confidence score: 0.9978
	Chunk: 

In [9]:

# for index, query_result in enumerate(chunked_results):
#     print(f"Query: {query_result['query']}")
#     print(f"True timestamp: {query_result['true_timestamp']}")
#
#
#     for chunk in query_result['predictions']:
#         print("\tChunk:", chunk[0])
#         print(f"\t\tClosest Moment: {chunk[1]['pred_relevant_windows'][0][0], chunk[1]['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {chunk[1]['pred_relevant_windows'][0][2]}")
#     print("-" * 50)

Now we have a problem, because we get a timestamp for each chunk, but we only need one timestamp for the whole video. The simplest solution is to pick confidence score, although this approach has a lot of shortcomings, we will use it, as alternatives require more complex algorithms

In [10]:
chunks_timestamp_results = []
for results_per_entry in results:

    max_results_per_entry = []
    for results_per_chunk in results_per_entry:
        max_score = 0.0
        best_timestamp = ()
        for chunk in results_per_chunk:
            prediction = chunk[1]
            if prediction['pred_relevant_windows'][0][2] > max_score:
                max_score = prediction['pred_relevant_windows'][0][2]
                best_timestamp = (prediction['pred_relevant_windows'][0][0] + 150 * chunk[0], prediction['pred_relevant_windows'][0][1] + 150 * chunk[0]) # we add 150 multiplied by chunk index to get the global timestamp(the model returns timestamp local to a particular chunk)
        max_results_per_entry.append(best_timestamp)

    chunks_timestamp_results.append(max_results_per_entry)

chunks_timestamp_results

[[(236.394, 250.0044),
  (206.5158, 218.892),
  (212.0606, 228.4743),
  (74.7838, 84.9939),
  (210.0026, 220.79430000000002),
  (209.4026, 227.7951)],
 [(228.76569999999998, 249.7944),
  (206.5393, 219.0424),
  (182.9217, 191.773),
  (207.5389, 221.4789),
  (250.8759, 263.37739999999997),
  (209.22129999999999, 227.8203)],
 [(217.75459999999998, 228.4726),
  (229.23950000000002, 249.66840000000002),
  (75.3822, 91.4614),
  (248.98930000000001, 255.4254),
  (206.9855, 221.96120000000002),
  (161.6167, 175.6835),
  (253.07479999999998, 262.7353),
  (190.2372, 202.2765),
  (172.676, 183.8671),
  (228.5769, 241.902)],
 [(217.91129999999998, 228.46030000000002),
  (241.049, 250.9572),
  (249.04430000000002, 255.19459999999998),
  (238.14499999999998, 250.2234),
  (189.6483, 202.48930000000001),
  (228.5798, 241.6187),
  (175.0452, 201.3957),
  (101.6467, 110.224),
  (190.2095, 202.387),
  (173.95420000000001, 201.4752)],
 [(120.0043, 130.1033),
  (241.33010000000002, 251.2347),
  (207.6551,

## Speeding up
Another approach is speeding up the video until it becomes short enough

In [11]:
output_path = "processed_videos/video_25_sped_up.mp4"
adjust_video_speed(og_video_path, output_path, 150)

Speeding up video by 1.83x...
Video is successfully sped up


In [18]:
sped_up_timestamps_results = []

for row in events.iterrows():
    queries, timestamps = row[1]
    predicted_timestamps = []


    results_per_entry = []
    for query in queries:
        print(f"Query: {query}")
        print(f"True timestamp: {query}")

        video = model.encode_video(output_path)
        prediction = model.predict(query, video)

        print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0], prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")

        results_per_entry.append((prediction['pred_relevant_windows'][0][0], prediction['pred_relevant_windows'][0][1]))
    print("-" * 50)

    sped_up_timestamps_results.append(results_per_entry)

Query: A man wearing headphones takes pictures with his camera while standing next to another person who has earphones plugged into it.
True timestamp: A man wearing headphones takes pictures with his camera while standing next to another person who has earphones plugged into it.
		Closest Moment: (0.7111, 7.0591), Corresponding Confidence score: 0.9925
Query: Two young men walk down an alleyway lined with shops, one wearing sunglasses and carrying a backpack, the other in orange shorts.
True timestamp: Two young men walk down an alleyway lined with shops, one wearing sunglasses and carrying a backpack, the other in orange shorts.
		Closest Moment: (0.6553, 5.4311), Corresponding Confidence score: 0.9842
Query: The two young men pass by parked cars before entering a building where they appear to be conversing or taking photos together.
True timestamp: The two young men pass by parked cars before entering a building where they appear to be conversing or taking photos together.
		Closest

In [19]:
# for query_result in results:
#     print(f"Query: {query_result['query']}")
#     print(f"True timestamp: {query_result['true_timestamp']}")
#
#     prediction = query_result['predictions']
#     print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0], prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")
#     print("-" * 50)

In [20]:
events_full['predicted_timestamps_chunks'] = np.array(chunks_timestamp_results)
events_full['predicted_timestamps_sped_up'] = np.array(sped_up_timestamps_results)

In [23]:
events_full.head()

,prompt_type,vlm_source,vlm_frame_count,parsed_events,predicted_timestamps_chunks,predicted_timestamps_sped_up
0,concise,long,30,[A man wearing headphones takes pictures with ...,"[(236.394, 250.0044), (206.5158, 218.892), (21...","[(0.7111, 7.0591), (0.6553, 5.4311), (0.0882, ..."
1,detailed,long,30,[A man takes pictures with his camera while an...,"[(228.76569999999998, 249.7944), (206.5393, 21...","[(1.0617, 6.8294), (0.6483, 5.3847), (0.0917, ..."
2,concise,medium,30,[A man is walking down an alleyway while weari...,"[(217.75459999999998, 228.4726), (229.23950000...","[(115.9076, 126.9845), (1.1871, 6.2375), (43.5..."
3,detailed,medium,30,[A man walks down an alleyway while wearing ea...,"[(217.91129999999998, 228.46030000000002), (24...","[(116.0874, 127.0461), (0.9166, 6.7191), (13.4..."
4,concise,short,30,[Aerial view of city buildings and streets tra...,"[(120.0043, 130.1033), (241.33010000000002, 25...","[(65.7385, 74.2094), (1.4454, 6.3432), (12.410..."


In [24]:
events_full.to_json(f"Timestamps/{og_video_path.split('/')[-1].split('.')[0]}_with_predictions.json", orient='records')